In [16]:
import sys
sys.path.append("..")

In [17]:
import tqdm
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [18]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, x_r, theta_0):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

In [40]:
GermanDataset(0).get_data(1)[0][0]

,duration,amount,age,personal_status_sex_1,personal_status_sex_2,personal_status_sex_3,personal_status_sex_5
993,1.252574,0.243766,-0.487784,0.0,0.0,0.0,1.0
859,-0.987573,0.108368,-0.839594,0.0,0.0,0.0,1.0
298,-0.240857,-0.268051,0.655598,0.0,0.0,0.0,1.0
553,-0.738668,-0.452361,-0.751642,0.0,0.0,0.0,1.0
672,3.243815,2.514684,0.567645,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...
835,-0.738668,-0.775968,1.095360,0.0,0.0,0.0,1.0
192,0.505858,0.228170,0.039930,0.0,0.0,0.0,1.0
629,-0.987573,0.198751,2.502599,0.0,0.0,0.0,1.0
559,-0.240857,-0.476109,-0.399832,0.0,0.0,0.0,1.0


In [19]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset):
    alpha = params['alpha']
    lamb = params['lamb']
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)
    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]
        x_r = recourse.get_recourse(x_0)
        append_result(results, recourse.name, seed, alpha, lamb, i, x_0, x_r, theta_0)

    df_results = pd.DataFrame(results)
    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f'../results/recourse/lr_{dataset.name}_{recourse.name}_{seed}.pkl')
    
    return df_results

In [20]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = LR()
        base_model.train(X_train.values, y_train.values)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        for recourse_fn in recourse_fns:
            recourse = recourse_fn(weights=weights_0, bias=bias_0, alpha=alpha)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset)
            results.append(df_results)

In [ ]:
torch.manual_seed(0)

d_results = {}
params = {}
params['alpha'] = 0.5 # float, None
params['lamb'] = 0.1
params['seeds'] = range(5)
params['save_results'] = True

datasets = [SyntheticDataset()]
recourse_fns = [LARRecourse, ROAR, ]

for dataset in datasets:
    results = []
    print(f'Running {dataset.name} data...')
    run_experiment(dataset, recourse_fns, params, results)
    
    d_results[dataset.name] = pd.concat(results)
    print(f'Finished {dataset.name}\n')

Running synthetic data...


[Alg1] [alpha=0.5] [lambda=0.1]: 100%|██████████| 96/96 [00:00<00:00, 26793.53it/s]


tensor([1.9661, 1.9713]) tensor([0.0506]) [1.9660925  1.9713432  0.05061606]


[ROAR] [alpha=0.5] [lambda=0.1]: 100%|██████████| 96/96 [00:55<00:00,  1.74it/s]

Finished synthetic



In [24]:
df = d_results["synthetic"]
df[df["i"]==95]


,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0
95,Alg1,0,0.5,0.1,95,"[-1.0356, -1.6137]","[0.0, 2.085]","[1.9661, 1.9713, 0.0506]"
95,ROAR,0,0.5,0.1,95,"[-1.0356, -1.6137]","[0.8281, 0.3923]","[1.9661, 1.9713, 0.0506]"
